In [68]:
import pandas as pd

In [69]:
tree_data = pd.read_csv(r'..\Data\new_york_tree_census_2015.csv')
tree_data['zipcode'].unique()

array([11366, 11370, 11434, 11209, 11692, 10031, 11207, 11219, 11421,
       10474, 11419, 10308, 10001, 10312, 11221, 10464, 11234, 10305,
       11201, 11233, 11411, 11220, 10009, 11372, 11432, 10314, 10461,
       10309, 11367, 11205, 10025, 11374, 11105, 11217, 10458, 11001,
       10459, 11364, 11379, 11691, 11004, 10032, 11358, 11238, 11213,
       11249, 10470, 11356, 11103, 11361, 10007, 11385, 11355, 11230,
       10469, 10468, 11426, 10304, 10456, 10463, 11206, 10462, 11216,
       11208, 10465, 11375, 11429, 11365, 11435, 11218, 10034, 11414,
       11412, 11377, 11223, 11427, 11204, 11357, 10022, 11215, 11428,
       10036, 10024, 10467, 10026, 11420, 11423, 10038, 11231, 11369,
       11413, 10453, 11210, 11436, 10307, 10475, 11229, 11363, 11040,
       10003, 10457, 11378, 11212, 11373, 10014, 11360, 11418, 11226,
       10301, 10473, 10472, 11235, 11225, 10302, 11422, 11354, 11236,
       11106, 10013, 11214, 10452, 10466, 10012, 11203,    83, 10310,
       11694, 10002,

In [70]:
tree_data_small = tree_data[['tree_dbh', 'curb_loc', 'status', 'health', 'spc_common', 'sidewalk', 'zipcode']].copy()
tree_data_small.head()

,tree_dbh,curb_loc,status,health,spc_common,sidewalk,zipcode
0,10,OnCurb,Alive,Good,green ash,NoDamage,11366
1,9,OnCurb,Alive,Good,honeylocust,NoDamage,11370
2,7,OnCurb,Alive,Good,Callery pear,NoDamage,11434
3,10,OnCurb,Alive,Good,Callery pear,NoDamage,11209
4,4,OnCurb,Alive,Good,'Schubert' chokecherry,NoDamage,11692


In [71]:
# initial clean of the data - drop na
tree_data_small.dropna(inplace=True)

tree_data_small.info()

<class 'pandas.core.frame.DataFrame'>
Index: 652166 entries, 0 to 683787
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   tree_dbh    652166 non-null  int64 
 1   curb_loc    652166 non-null  object
 2   status      652166 non-null  object
 3   health      652166 non-null  object
 4   spc_common  652166 non-null  object
 5   sidewalk    652166 non-null  object
 6   zipcode     652166 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 39.8+ MB


In [72]:
def get_ratio(series):
       val_counts = series.value_counts()
       if len(val_counts) > 1:
              return val_counts.iloc[0]/ val_counts.iloc[1]
       # if there is only one value then the ratio is 1
       return 1

def get_top_3(df, zip_col_name, split_col_name):
       # '''
       # Takes in the full dataframe and returns a dataframe of zip code and the top 3 most common values for split_col_name 
       # :param series: 
       # :return: 
       # '''
       df.groupby(zip_col_name)
       val_counts = df[split_col_name].value_counts()
       ret_df = df[zip_col_name]
       if len(val_counts) >= 3:
              ret_df[split_col_name + '1'] = val_counts[0]
              ret_df[split_col_name + '2'] = val_counts[1]
              ret_df[split_col_name + '3'] = val_counts[2]
           
def get_first(series):
       val_counts = series.value_counts()
       return val_counts.index[0]

def get_second(series):
       val_counts = series.value_counts()
       if len(val_counts) > 1:
              return val_counts.index[1]
       else:
              return val_counts.index[-1]

def get_third(series):
       val_counts = series.value_counts()
       if len(val_counts) > 2:
              return val_counts.index[2]
       else:
              return val_counts.index[-1]
       
def count_unique_vals(series):
       return series.value_counts().to_dict()
       

In [73]:
# because we are going to join on zip code, we need to make meaningful aggregates for each column per zip code
tree_data_final = tree_data_small.groupby('zipcode').agg(
       dbh_mean=('tree_dbh', 'mean'),
       curb_ratio=('curb_loc', get_ratio),
       status_ratio=('status', get_ratio),
       sidewalk_ratio=('sidewalk', get_ratio),
       health_counts=('health', count_unique_vals),
       species_1=('spc_common', get_first),
       species_2=('spc_common', get_second),
       species_3=('spc_common', get_third)                          
).reset_index()

health_counts_df = pd.DataFrame(tree_data_final['health_counts'].tolist()).fillna(0).astype(int)
tree_data_final = pd.concat([tree_data_final, health_counts_df], axis=1).drop(columns=['health_counts'])

tree_data_final


,zipcode,dbh_mean,curb_ratio,status_ratio,sidewalk_ratio,species_1,species_2,species_3,Good,Fair,Poor
0,83,16.505365,1.273171,1,3.217195,American elm,pin oak,ginkgo,885,39,8
1,10001,7.365882,46.222222,1,4.629139,honeylocust,Callery pear,Japanese zelkova,719,100,31
2,10002,8.459222,5.255072,1,4.678947,London planetree,honeylocust,ginkgo,1656,387,115
3,10003,9.077200,91.523810,1,2.672968,honeylocust,Callery pear,ginkgo,1480,361,102
4,10004,6.658120,3.178571,1,28.250000,honeylocust,ginkgo,Japanese zelkova,97,17,3
...,...,...,...,...,...,...,...,...,...,...,...
186,11691,8.813316,27.221053,1,4.432624,honeylocust,London planetree,cherry,3761,1258,343
187,11692,4.962963,12.027972,1,5.900000,Callery pear,cherry,Sophora,1122,523,218
188,11693,7.897335,6.914062,1,12.876712,London planetree,honeylocust,Callery pear,518,288,207
189,11694,8.053841,17.358382,1,6.234624,honeylocust,Callery pear,cherry,1958,833,385


In [74]:
crime_data = pd.read_csv(r'..\Data\2015_Crime.csv', low_memory=False)

In [75]:
crime_data.head()

,Ticket Number,Violation Date,Violation Time,Issuing Agency,Respondent First Name,Respondent Last Name,Balance Due,Violation Location (Borough),Violation Location (Block No.),Violation Location (Lot No.),...,Charge #8: Code Description,Charge #8: Infraction Amount,Charge #9: Code,Charge #9: Code Section,Charge #9: Code Description,Charge #9: Infraction Amount,Charge #10: Code,Charge #10: Code Section,Charge #10: Code Description,Charge #10: Infraction Amount
0,0187818290,06/30/2015,07:50:00,SANITATION OTHERS,NaN,607 2 AVE RE,0.0,MANHATTAN,914.0,28.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1276215F1,06/30/2015,NaN,DOHMH - BFSCS,NaN,JEM PROVISIONS INC,NaN,BROOKLYN,NaN,NaN,...,Toilet facility not properly supplied in that ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0700053870,06/30/2015,13:54:00,DEPT OF TRAN,NaN,CONSOLIDATED EDISON,NaN,QUEENS,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0700051990,06/30/2015,14:10:00,DEPT OF TRAN,NaN,CONSOLIDATED EDISON,NaN,QUEENS,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0700051981,06/30/2015,14:04:00,DEPT OF TRANSPORTATION,NaN,CONSOLIDATED EDISON,NaN,QUEENS,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [76]:
crime_data.columns

Index(['Ticket Number', 'Violation Date', 'Violation Time', 'Issuing Agency',
       'Respondent First Name', 'Respondent Last Name', 'Balance Due',
       'Violation Location (Borough)', 'Violation Location (Block No.)',
       'Violation Location (Lot No.)', 'Violation Location (House #)',
       'Violation Location (Street Name)', 'Violation Location (Floor)',
       'Violation Location (City)', 'Violation Location (Zip Code)',
       'Violation Location (State Name)', 'Respondent Address (Borough)',
       'Respondent Address (House #)', 'Respondent Address (Street Name)',
       'Respondent Address (City)', 'Respondent Address (Zip Code)',
       'Respondent Address (State Name)', 'Hearing Status', 'Hearing Result',
       'Scheduled Hearing Location', 'Hearing Date', 'Hearing Time',
       'Decision Location (Borough)', 'Decision Date',
       'Total Violation Amount', 'Violation Details', 'Date Judgment Docketed',
       'Respondent Address or Facility Number(For FDNY and DOB Ti

In [77]:
crime_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750352 entries, 0 to 750351
Data columns (total 78 columns):
 #   Column                                                           Non-Null Count   Dtype  
---  ------                                                           --------------   -----  
 0   Ticket Number                                                    750352 non-null  object 
 1   Violation Date                                                   750352 non-null  object 
 2   Violation Time                                                   611192 non-null  object 
 3   Issuing Agency                                                   631478 non-null  object 
 4   Respondent First Name                                            348156 non-null  object 
 5   Respondent Last Name                                             701497 non-null  object 
 6   Balance Due                                                      188781 non-null  float64
 7   Violation Location (Borough) 

In [78]:
crime_data = crime_data.dropna(subset=['Ticket Number', 'Violation Date', 'Violation Time', 'Issuing Agency',
       'Respondent First Name', 'Respondent Last Name', 'Balance Due',
       'Violation Location (Borough)', 'Violation Location (Block No.)',
       'Violation Location (Lot No.)', 'Violation Location (House #)',
       'Violation Location (Street Name)', 'Violation Location (Floor)',
       'Violation Location (City)', 'Violation Location (Zip Code)',
       'Violation Location (State Name)', 'Respondent Address (Borough)',
       'Respondent Address (House #)', 'Respondent Address (Street Name)',
       'Respondent Address (City)', 'Respondent Address (Zip Code)',
       'Respondent Address (State Name)', 'Hearing Status', 'Hearing Result',
       'Scheduled Hearing Location', 'Hearing Date', 'Hearing Time',
       'Decision Location (Borough)', 'Decision Date',
       'Total Violation Amount', 'Violation Details', 'Date Judgment Docketed',
       'Respondent Address or Facility Number(For FDNY and DOB Tickets)',
       'Penalty Imposed', 'Paid Amount', 'Additional Penalties or Late Fees',
       'Compliance Status', 'Violation Description', 'Charge #1: Code',
       'Charge #1: Code Section', 'Charge #1: Code Description',
       'Charge #1: Infraction Amount', 'Charge #2: Code',
       'Charge #2: Code Section', 'Charge #2: Code Description',
       'Charge #2: Infraction Amount', 'Charge #3: Code',
       'Charge #3: Code Section', 'Charge #3: Code Description',
       'Charge #3: Infraction Amount', 'Charge #4: Code',
       'Charge #4: Code Section', 'Charge #4: Code Description',
       'Charge #4: Infraction Amount', 'Charge #5: Code',
       'Charge #5: Code Section', 'Charge #5: Code Description',
       'Charge #5: Infraction Amount', 'Charge #6: Code',
       'Charge #6: Code Section', 'Charge #6: Code Description',
       'Charge #6: Infraction Amount', 'Charge #7: Code',
       'Charge #7: Code Section', 'Charge #7: Code Description',
       'Charge #7: Infraction Amount', 'Charge #8: Code',
       'Charge #8: Code Section', 'Charge #8: Code Description',
       'Charge #8: Infraction Amount', 'Charge #9: Code',
       'Charge #9: Code Section', 'Charge #9: Code Description',
       'Charge #9: Infraction Amount', 'Charge #10: Code',
       'Charge #10: Code Section', 'Charge #10: Code Description',
       'Charge #10: Infraction Amount'])

In [79]:
crime_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 78 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   Ticket Number                                                    0 non-null      object 
 1   Violation Date                                                   0 non-null      object 
 2   Violation Time                                                   0 non-null      object 
 3   Issuing Agency                                                   0 non-null      object 
 4   Respondent First Name                                            0 non-null      object 
 5   Respondent Last Name                                             0 non-null      object 
 6   Balance Due                                                      0 non-null      float64
 7   Violation Location (Borough)                                 